In [ ]:
#Upload dataset
# import required packages
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
from tqdm import tqdm


In [ ]:
#Data exploration: look at number of unique entries for some of the columns

categorical_cols = game_df.select_dtypes(include=['object', 'category'])
for col in categorical_cols:
    print(f"--- Unique Values for: {col} ---")
    print(game_df[col].unique())
    print("\n") # Adds a space for readability

In [ ]:
# select relevant variables, excluding Game Title, User Review Text, Release Year
selected_var = ['User Rating', 'Age Group Targeted', 'Price', 'Platform', 'Requires Special Device', 'Developer', 'Publisher', 'Genre','Multiplayer',
            'Game Length (Hours)','Graphics Quality', 'Soundtrack Quality','Story Quality','Game Mode','Min Number of Players']
game_df = game_df[selected_var]

In [ ]:
# List of columns that are categorical with multiple possible categories
multi_cat_cols = ['Age Group Targeted', 'Platform', 'Developer', 'Publisher',
                  'Genre', 'Graphics Quality', 'Soundtrack Quality', 'Story Quality']

# Boolean columns (True/False)
bool_cols = ['Requires Special Device', 'Multiplayer', 'Game Mode']

# ---- STEP 1: Ensure the boolean columns are actually booleans ----
for col in bool_cols:
    game_df[col] = game_df[col].astype(bool)

# ---- STEP 2: Convert boolean columns to integers (0 and 1) ----
# This keeps them usable for machine learning models without expanding into dummy columns.
for col in bool_cols:
    game_df[col] = game_df[col].astype(int)

# ---- STEP 3: Apply one-hot encoding to the multi-category columns ----
game_df = pd.get_dummies(game_df, columns=multi_cat_cols, drop_first=False)

# ---- STEP 4: Inspect the result ----
game_df.head()

In [ ]:
#Define X and Y
# Separate predictors (X) and target (y)
# -------------------------------
y = game_df[['Price']]
X = game_df.drop(columns=['Price'])

print(X.shape, y.shape)

In [ ]:
#Train/Test Split
# -------------------------------
# 80% training, 20% testing
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

print("Training:", X_train.shape, "Testing:", X_test.shape)

In [ ]:
# -------------------------------
# Scale predictors using StandardScaler
# -------------------------------
scaler = StandardScaler()

scaler.fit(X_train)      # fit on training only
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# -------------------------------
# Try k = 1 first
# -------------------------------
knn = KNeighborsRegressor(n_neighbors=1)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

# Fix: Remove 'squared=False' and use np.sqrt for RMSE
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R²:", r2)

In [ ]:
# -------------------------------
# Loop through k from 1 to 50
# -------------------------------
results = []

for k in tqdm(range(1, 51)):
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)

    y_pred_k = knn.predict(X_test_scaled)

    # MSE first, then take square root for RMSE
    rmse_k = mean_squared_error(y_test, y_pred_k) ** 0.5

    results.append({'k': k, 'rmse': rmse_k})

results_df = pd.DataFrame(results)
results_df

In [ ]:
#visualize RMSE vs K
results_df.plot.scatter(x='k', y='rmse', figsize=(8,5))
plt.title("RMSE vs k for KNN Regressor")
plt.show()

In [ ]:
# get the best K
best_k = results_df.loc[results_df['rmse'].idxmin()]
best_k


In [ ]:

final_k = int(best_k['k'])
print("Best k =", final_k)

knn_final = KNeighborsRegressor(n_neighbors=final_k)
knn_final.fit(X_train_scaled, y_train)

y_pred_final = knn_final.predict(X_test_scaled)

    # MSE first, then take square root for RMSE
rmse_final = mean_squared_error(y_test, y_pred_final) ** 0.5
r2_final = r2_score(y_test, y_pred_final)

print("Final RMSE:", rmse_final)
print("Final R²:", r2_final)

In [ ]:
# Attach predictions to test set
test_results = X_test.copy()
test_results['Actual Price'] = y_test.values
test_results['Predicted Price'] = y_pred_final

# Sort by predicted price to find high-profit games
test_results.sort_values(by='Predicted Price', ascending=False).head(10)